# MODEL FEATURE SELECTION
In this notebook, I will be drawing correlation matrices for each dataframe that I obtained from my fresh_start.ipynb file so that we can start forecasting for each industry.

In [43]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Import dataframes and labels

In [44]:
openings_X = pd.read_csv('data/reduced/openings_X_data.csv')
layoffs_X = pd.read_csv('data/reduced/layoffs_X_data.csv')
openings_y = pd.read_csv('data/labels/openings_y_data.csv')
layoffs_y = pd.read_csv('data/labels/layoffs_y_data.csv')

## Find correlations between the labels and the features in openings dfs

In [45]:
openings_X = openings_X.drop(columns=['Unnamed: 0'])
layoffs_X = layoffs_X.drop(columns=['Unnamed: 0'])
openings_y = openings_y.drop(columns=['Unnamed: 0'])
layoffs_y = layoffs_y.drop(columns=['Unnamed: 0'])

In [46]:
print(list(openings_X.columns))
print(list(openings_y.columns))

['Hires: Total nonfarm in Ohio - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)', 'Hires: Total nonfarm in Oregon - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)', 'Hires: Total nonfarm in Pennsylvania - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)', 'Hires: Total nonfarm in Rhode Island - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)', 'Hires: Total nonfarm in South Carolina - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)', 'Hires: Total nonfarm in South Dakota - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)', 'Hires: Total nonfarm in Tennessee - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)', 'Hires: Total nonfarm in Texas - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)', 'Hires: Total nonfarm in Utah - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)', 'Hires: Total nonfarm in Vermont - Rate in Percent, Monthly, Seaso

In [47]:
def compute_lagged_spearman_top10(X, Y, lags=[0]):
    top_features_dict = {}

    for label in Y.columns:
        lagged_results = []

        for lag in lags:
            Y_lagged = Y[label].shift(-lag)  # shift up to align past X with future Y
            aligned = pd.concat([X, Y_lagged.rename(label)], axis=1).dropna()
            X_aligned = aligned[X.columns]
            Y_aligned = aligned[label]

            corr_series = X_aligned.corrwith(Y_aligned, method='spearman')
            top_corrs = corr_series.abs().sort_values(ascending=False).head(10)

            lagged_results.append({
                'lag': lag,
                'top_features': {
                    feature: corr_series[feature] for feature in top_corrs.index
                }
            })

        top_features_dict[label] = lagged_results

    return top_features_dict

In [48]:
# Initialize dictionary to store top features and their correlations per label
top_features_dict_openings = compute_lagged_spearman_top10(openings_X, openings_y, [0,1,2,3])

In [49]:
top_features_dict_openings

{'Job Openings: Total nonfarm - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': [{'lag': 0,
   'top_features': {'Hires: State and local government, excluding education - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(0.7005101614036647),
    'Hires: Total nonfarm in South Carolina - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(0.655223512331025),
    'Quits: Private educational services - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(0.6478137598666164),
    'Hires: State and local government education - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(0.6239317604219325),
    'Hires: Construction - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(-0.621695060090696),
    'Hires: Total nonfarm in Tennessee - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(0.6041980168349715),
    'Hires

In [50]:
# Initialize dictionary to store top features and their correlations per label
top_features_dict_layoffs = compute_lagged_spearman_top10(layoffs_X, layoffs_y, [0,1,2,3])

In [51]:
top_features_dict_layoffs

{'Layoffs and Discharges: Total nonfarm - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': [{'lag': 0,
   'top_features': {'Layoffs and Discharges: Total nonfarm in South Carolina - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(0.7000176078863036),
    'Job Openings: Finance and insurance - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(-0.68305399988871),
    'Job Openings: Real estate and rental and leasing - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(-0.6240886674546816),
    'Job Openings: Federal - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(-0.5493390958841471),
    'Hires: Construction - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(0.5242811626118538),
    'Quits: Mining and logging - Rate in Percent, Monthly, Seasonally Adjusted (All size classes)': np.float64(-0.5086203689115386),
    'Quits: Priv

## Save the dictionaries for use later!

In [52]:
import pickle

with open("data/MODELING_FEATURE_NAMES/top_features_openings.pkl", "wb") as f:
    pickle.dump(top_features_dict_openings, f)

with open("data/MODELING_FEATURE_NAMES/top_features_layoffs.pkl", "wb") as f:
    pickle.dump(top_features_dict_layoffs, f)